In [0]:
ls /Volumes/workspace/sp100_stock_price/sp100data/

In [0]:
# Reads CSV data stored in a DBFS using the dataframe
df = spark.read.format("csv").option("header", True).load(
    "/Volumes/workspace/sp100_stock_price/sp100data/sp100_historical_data.csv"
)
display(df)

In [0]:
# Lists all companies available on the dataset using pandas
tickers = [row['ticker'] for row in df.select('ticker').distinct().collect()]
display(tickers)

In [0]:
#Reads list of companies in full
df.createOrReplaceTempView("sp100")
names_df = spark.sql("select distinct name from sp100")
display(names_df)

In [0]:
df.dtypes

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, lag, round

window_spec = Window.partitionBy("ticker").orderBy("date")

df_casted = df.withColumn("close", col("close").cast("double"))

df_with_prev = df_casted.withColumn(
    "prev_close", lag(col("close")).over(window_spec)
)

df_with_return = df_with_prev.withColumn(
    "daily_return_pct",
    round(
        ((col("close") - col("prev_close")) / col("prev_close")) * 100, 4
    )
)

display(df_with_return)

In [0]:
from pyspark.sql.functions import year, max as spark_max, min as spark_min, col

df_with_year = df_with_return.withColumn("year", year(col("date")))

high_low_df = (
    df_with_year.groupBy("ticker", "year")
    .agg(
        spark_max("close").alias("highest_close"),
        spark_min("close").alias("lowest_close")
    )
)

display(high_low_df)

In [0]:
df_2025 = high_low_df.filter(col("year") == 2025)
display(df_2025)

In [0]:
# Join to get all columns for highest close
highest_df = df_with_year.join(
    high_low_df,
    (df_with_year["ticker"] == high_low_df["ticker"]) &
    (df_with_year["year"] == high_low_df["year"]) &
    (df_with_year["highest_close"] == high_low_df["highest_close"])
).select(df_with_year["*"])

# Join to get all columns for lowest close
lowest_df = df_with_year.join(
    high_low_df,
    (df_with_year["ticker"] == high_low_df["ticker"]) &
    (df_with_year["year"] == high_low_df["year"]) &
    (df_with_year["lowest_close"] == high_low_df["lowest_close"])
).select(df_with_year["*"])

# Union both DataFrames and remove duplicates if a row is both highest and lowest
result_df = highest_df.unionByName(lowest_df).dropDuplicates()

display(result_df)